# Block-wise DVS motion compensation on DSEC

This notebook implements and evaluates overlapping block-wise event translation on a free CPU runtime. Local affine and homography are intentionally out of scope.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/itsvvy/dsec-event-motion-compensation.git"
BRANCH = "fix/blockwise-phase-initialization"
REPO_DIR = Path("/content/dsec-event-motion-compensation")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

%cd /content/dsec-event-motion-compensation
sys.path.insert(0, str(REPO_DIR))
%pip install -q -r requirements.txt

In [ ]:
from tests.test_synthetic import test_recovers_known_constant_translation
from tests.test_blockwise import (
    test_phase_correlation_recovers_content_motion,
    test_blockwise_estimation_recovers_two_motion_regions,
    test_short_baseline_resolves_spurious_zero_long_peak,
)

test_recovers_known_constant_translation()
test_phase_correlation_recovers_content_motion()
test_blockwise_estimation_recovers_two_motion_regions()
test_short_baseline_resolves_spurious_zero_long_peak()
print("All synthetic motion-compensation tests passed.")

## Run DSEC full-frame experiment

A `64 x 64` tile and 32 px stride are used to preserve local driving-scene motion. The comparison uses every valid DSEC flow pixel rather than a hand-selected patch median.

In [ ]:
%env MPLCONFIGDIR=/content/.matplotlib
%time !python scripts/run_blockwise_demo.py --data-dir /content/dsec_data --output-dir /content/dsec_blockwise --flow-indices 2 8 12 --tile-size 64 --stride 32

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

results_dir = Path("/content/dsec_blockwise")
summary = json.loads((results_dir / "summary.json").read_text())
table = pd.DataFrame(summary["results"])
columns = [
    "flow_index", "window_ms", "event_count", "valid_blocks", "total_blocks",
    "global_median_endpoint_error_px", "blockwise_median_endpoint_error_px",
    "global_within_2px", "blockwise_within_2px", "warp_coverage", "runtime_s",
]
display(table[columns].round(3))
display(Image(filename=str(results_dir / "benchmark_summary.png"), width=1200))
for figure in sorted(results_dir.glob("blockwise_f*.png")):
    display(Image(filename=str(figure), width=1200))

## Interpretation

Block-wise translation resolves the full-frame limitation of a single global vector and improves dense DSEC-flow agreement substantially. The remaining lower-left high-texture error demonstrates that local translation is still only a baseline. Rotation and scale residuals become the next target for local affine; homography remains deferred to selected planar regions.